# Stage 1 — GPU pipeline (Qwen3-8B)

Extract activations, build axis, run AUROC gate.

- **Runtime:** A100 GPU
- **Preset `default`:** faithful path (0.93 gate, `value_axis.npy`)
- **Preset `dev`:** Qwen-local ICRL on this runtime (0.75 gate, `value_axis_proxy.npy`)

In [ ]:
PRESET = 'dev'  # 'default' | 'dev'
N = 100         # ICRL conversations (dev preset only)

import torch
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))

In [ ]:
import os
if not os.path.isdir('/content/latent_failiure_prediction'):
    !git clone https://github.com/abdelmagid07/latent_failiure_prediction.git
else:
    %cd /content/latent_failiure_prediction && git pull origin main
%cd /content/latent_failiure_prediction/stage1
!pip install -q -e .

In [ ]:
from pathlib import Path
from stage1.common.paths import data_file

if PRESET == 'dev':
    ICRL = data_file('icrl_proxy.json')
    ACT_DIR = 'data/activations_proxy'
    !python -m stage1.icrl_gen.generate --n {N} --backend local_qwen \
      --output {ICRL} --resume --syntactic-only
else:
    from google.colab import files
    ICRL = data_file('icrl.json')
    ACT_DIR = 'data/activations'
    if not ICRL.exists():
        up = files.upload()
        Path(next(iter(up))).rename(ICRL)
print('ICRL:', ICRL)

In [ ]:
!python -m stage1.pipeline.extract_activations --icrl {ICRL} \
  --activations-dir {ACT_DIR} --force

In [ ]:
import subprocess, sys
args = (['--preset', 'dev', '--icrl', str(ICRL), '--skip-extract'] if PRESET == 'dev'
        else ['--icrl', str(ICRL), '--skip-mock', '--skip-extract'])
subprocess.run([sys.executable, '-m', 'stage1.pipeline.run_gate', *args], check=True)

In [ ]:
from google.colab import files
import json
from stage1.common.paths import data_file

axis = data_file('value_axis_proxy.npy' if PRESET == 'dev' else 'value_axis.npy')
manifest = data_file('axis_manifest_proxy.json' if PRESET == 'dev' else 'axis_manifest.json')
for p in [axis, manifest, data_file('auroc_by_layer_proxy.png' if PRESET == 'dev' else 'auroc_by_layer.png')]:
    if p.exists():
        files.download(str(p))